# 1 — Local first look

**Snapshot:** open the files, understand their shape, and prepare a schema-only AI prompt.

The two assessment files in this demo are **synthetic**. The workflow is designed for a more sensitive setting:

- Python reads the CSV files on this computer.
- This notebook makes **no network or AI API calls**.
- Names and student IDs are not displayed.
- Any optional LLM receives a description of the schema and task—not student records.

> Local processing is not automatically policy-compliant. Use an approved device and environment, and follow your organization's rules for student data.

> **Opener:** If you have not seen `00_research_wow.ipynb` yet, start there—it reproduces a landmark education-research finding live and sets up the story this notebook continues.

## The boundary we want to demonstrate

**Student-level data stays local.** An LLM can help draft generic pandas code from field names and requirements; a person reviews that code; Jupyter runs it against local data.

`local CSV → local inspection → schema-only prompt → reviewed code → local execution`

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)


def locate_data_file(filename):
    """Find a data file when the notebook starts in /notebooks or the project root."""
    for folder in (Path.cwd(), Path.cwd().parent):
        candidate = folder / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r} in the current folder or its parent.")


reading_path = locate_data_file("FakeiReadyData.csv")
math_path = locate_data_file("FakeiReadyMathData.csv")

# Read identifiers as strings so leading zeros would be preserved in a real export.
reading_raw = pd.read_csv(
    reading_path,
    dtype={"Student Id": "string", "Class": "string"},
)
math_raw = pd.read_csv(
    math_path,
    dtype={"Student Id": "string", "Class": "string"},
)

## Load confirmation (aggregate metadata only)

In [2]:
file_overview = pd.DataFrame(
    {
        "file": [reading_path.name, math_path.name],
        "rows": [len(reading_raw), len(math_raw)],
        "columns": [reading_raw.shape[1], math_raw.shape[1]],
        "missing_cells": [
            int(reading_raw.isna().sum().sum()),
            int(math_raw.isna().sum().sum()),
        ],
        "duplicate_student_ids": [
            int(reading_raw["Student Id"].duplicated().sum()),
            int(math_raw["Student Id"].duplicated().sum()),
        ],
    }
)
file_overview

,file,rows,columns,missing_cells,duplicate_student_ids
0,FakeiReadyData.csv,60,83,0,0
1,FakeiReadyMathData.csv,60,62,0,0


## A safe preview for this synthetic demo

The raw dataframes contain names and IDs, but the preview deliberately creates temporary aliases and selects only a few fields. Aliases reduce casual exposure; they do **not** make real student data anonymous.

In [3]:
safe_preview = (
    reading_raw.assign(
        student_key=[f"S{number:03d}" for number in range(1, len(reading_raw) + 1)]
    )[
        ["student_key", "Current Grade", "Class", "Gender", "SWD", "MLL", "ED"]
    ]
    .rename(
        columns={
            "Current Grade": "grade",
            "Class": "class_section",
            "Gender": "gender",
            "SWD": "swd",
            "MLL": "mll",
            "ED": "economically_disadvantaged",
        }
    )
)
safe_preview.head(8)

,student_key,grade,class_section,gender,swd,mll,economically_disadvantaged
0,S001,6,601,F,Yes,No,Yes
1,S002,6,602,M,No,No,Yes
2,S003,6,601,F,No,Yes,No
3,S004,6,602,M,No,No,Yes
4,S005,6,601,F,No,No,No
5,S006,6,602,M,No,No,Yes
6,S007,6,601,F,No,No,Yes
7,S008,6,602,M,Yes,No,No


## Inspect the assessment structure without inspecting records

In [4]:
def assessment_inventory(frame):
    fields = []
    for column in frame.columns:
        if not column.startswith("I-Ready Result"):
            continue
        parts = column.split("\n")
        fields.append(
            {
                "subject": parts[1],
                "field": parts[2],
                "school_year": parts[3],
            }
        )
    return pd.DataFrame(fields)


inventory = pd.concat(
    [assessment_inventory(reading_raw), assessment_inventory(math_raw)],
    ignore_index=True,
)
inventory[
    inventory["field"].str.contains("Overall SS|Overall Placement|Percentile")
].drop_duplicates().reset_index(drop=True)

,subject,field,school_year
0,Reading,Overall SS - Fall,2025-2026
1,Reading,Overall Placement - Fall,2025-2026
2,Reading,Percentile - Fall,2025-2026
3,Reading,Comprehension: Overall Placement - Fall,2025-2026
4,Reading,Overall SS - Winter,2025-2026
...,...,...,...
16,Math,Overall Placement - Winter,2025-2026
17,Math,Percentile - Winter,2025-2026
18,Math,Overall SS - Spring,2025-2026
19,Math,Overall Placement - Spring,2025-2026


## Prompt 1 — ask for preparation code, not an analysis of records

The next cell *constructs text*. It does not contact an LLM. Read the output before copying it into an **organization-approved** assistant. Notice that the prompt describes dataframe names, fields, and safeguards but contains no rows or student values.

In [5]:
prompt_1 = """
I am using pandas in a local Jupyter notebook. The data may contain student PII,
so do not ask me to upload files, paste rows, or provide names/IDs. Write code
against the dataframe and column names below; the code will run locally.

Existing dataframes:
- reading_raw: one row per student, Reading results
- math_raw: one row per student, Math results
- Join key: "Student Id" (a direct identifier; use only for validation/joining)
- Direct identifiers: "Student Id", "Student Name" (never display or export)
- Shared fields: Gender, SWD, ED, Current Grade, Class
- Reading calls the multilingual-learner field MLL; Math calls it LEP
- Seasons: Fall, Winter, Spring
- Needed measures per subject/season: Overall SS, Percentile, Overall Placement
- Assessment headings follow:
  I-Ready Result\\n{Subject}\\n{Measure} - {Season}\\n2025-2026

Return one self-contained Python code block that:
1. verifies IDs are nonmissing and unique in each file;
2. verifies the ID sets and shared demographic fields agree before joining;
3. creates session-only aliases S001, S002, ...;
4. builds `analysis`, one row per alias and season, with reading/math score,
   percentile, placement, and renamed demographic fields;
5. removes direct identifiers from `analysis`;
6. asserts the expected row count, uniqueness, three seasons per student,
   and complete overall scores.

Constraints: pandas only; no network/API calls; no file writes; no display of
names or IDs. Explain that aliases are pseudonyms, not anonymization.
""".strip()

print(prompt_1)

I am using pandas in a local Jupyter notebook. The data may contain student PII,
so do not ask me to upload files, paste rows, or provide names/IDs. Write code
against the dataframe and column names below; the code will run locally.

Existing dataframes:
- reading_raw: one row per student, Reading results
- math_raw: one row per student, Math results
- Join key: "Student Id" (a direct identifier; use only for validation/joining)
- Direct identifiers: "Student Id", "Student Name" (never display or export)
- Shared fields: Gender, SWD, ED, Current Grade, Class
- Reading calls the multilingual-learner field MLL; Math calls it LEP
- Seasons: Fall, Winter, Spring
- Needed measures per subject/season: Overall SS, Percentile, Overall Placement
- Assessment headings follow:
  I-Ready Result\n{Subject}\n{Measure} - {Season}\n2025-2026

Return one self-contained Python code block that:
1. verifies IDs are nonmissing and unique in each file;
2. verifies the ID sets and shared demographic fields a

### What to copy back

1. Review the LLM response for network calls, file writes, or identifier output.
2. Copy **only the Python inside its fenced code block** into a new code cell in a duplicate notebook.
3. Run the assertions before any analysis.
4. Compare with the reviewed example in **`02_clean_analysis_table.ipynb`**.

Never paste an error message back into an external assistant until you have checked that it contains no record values, local paths, names, or IDs.